In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q transformers[sentencepiece] datasets rouge-score bert-score accelerate

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


### Load and Explore the Bengali XL-Sum Dataset
We will define the dataset path and load the JSONL files (typically train, validation, and test splits are present in the XL-Sum folder).

In [2]:
import os
import json
import pandas as pd
from datasets import Dataset, DatasetDict

dataset_dir = "/content/drive/MyDrive/Colab Notebooks/CL-NLP - Lab/lab-8/bengali_XLSum_v2.0/"

def load_xlsum_jsonl(file_path):
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    return Dataset.from_pandas(pd.DataFrame(data))

# Dynamically find the jsonl files
files = os.listdir(dataset_dir)
train_file = os.path.join(dataset_dir, [f for f in files if 'train' in f.lower()][0])
val_file = os.path.join(dataset_dir, [f for f in files if 'val' in f.lower() or 'dev' in f.lower()][0])
test_file = os.path.join(dataset_dir, [f for f in files if 'test' in f.lower()][0])

raw_datasets = DatasetDict({
    'train': load_xlsum_jsonl(train_file),
    'validation': load_xlsum_jsonl(val_file),
    'test': load_xlsum_jsonl(test_file)
})

print("Dataset Structure:")
print(raw_datasets)

# Display sample sample
print("\nSample Data:")
print("Text:", raw_datasets['test'][0]['text'][:300] + "...")
print("Summary:", raw_datasets['test'][0]['summary'])


Dataset Structure:
DatasetDict({
    train: Dataset({
        features: ['id', 'url', 'title', 'summary', 'text'],
        num_rows: 8102
    })
    validation: Dataset({
        features: ['id', 'url', 'title', 'summary', 'text'],
        num_rows: 1012
    })
    test: Dataset({
        features: ['id', 'url', 'title', 'summary', 'text'],
        num_rows: 1012
    })
})

Sample Data:
Text: আটক স্বজনদের আইডি কার্ড হাতে উইগুর নারীদের বিক্ষোভ (ফাইল ফটো) অভিযোগ করেন জাতিসংঘের জাতিগত বৈষম্য বিষয়ক কমিটির সদস্য গে ম্যাকডুগাল। তিনি বলেন, এমন খবর তিনি পেয়েছেন যে চীনা কর্তৃপক্ষ স্বায়ত্তশাসিত উইগুর প্রদেশকে কার্যত "বিশাল একটি বন্দীশিবিরে" রূপান্তরিত করেছে। চীন সাথে সাথে এর কোনো জবাব দেয়নি। ৫...
Summary: জাতিসংঘের মানবাধিকার বিষয়ক কমিটি জেনেভাতে চীনের ওপর দুদিনের যে বিশেষ সভা করছে, সেখানে অভিযোগ করা হয়, কট্টরপন্থী সন্দেহে ১০ লাখ চীনা উইগুর মুসলিমকে বিভিন্ন রাজনৈতিক শিবিরে আটকে রাখা হয়েছে।


### Evaluate mT5 Model
We will load the pre-trained `google/mt5-small` model and generate abstractive summaries for a sample test subset to compare with the reference summaries.

In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

device = "cuda" if torch.cuda.is_available() else "cpu"
# Load a checkpoint of mT5 fine-tuned on XL-Sum (which includes Bengali)
mt5_model_name = "csebuetnlp/mT5_multilingual_XLSum"

print(f"Loading tokenizer and model for {mt5_model_name} on {device}...")
mt5_tokenizer = AutoTokenizer.from_pretrained(mt5_model_name)
mt5_model = AutoModelForSeq2SeqLM.from_pretrained(mt5_model_name).to(device)

def generate_summary_mt5(text, max_len=64):
    # This model is pre-trained directly on XL-Sum text structures
    inputs = mt5_tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(device)
    outputs = mt5_model.generate(
        inputs["input_ids"],
        max_length=max_len,
        num_beams=4,
        repetition_penalty=2.5,
        no_repeat_ngram_size=3,
        early_stopping=True
    )
    return mt5_tokenizer.decode(outputs[0], skip_special_tokens=True)

# select 10 samples for quick evaluation
eval_subset = raw_datasets['test'].select(range(10))

mt5_predictions = []
references = []

for i, sample in enumerate(eval_subset):
    pred = generate_summary_mt5(sample['text'])
    mt5_predictions.append(pred)
    references.append(sample['summary'])
    print(f"Sample {i+1} Generated Summary (mT5):", pred)


Loading tokenizer and model for csebuetnlp/mT5_multilingual_XLSum on cuda...


config.json:   0%|          | 0.00/730 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/375 [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B / 4.31MB            

spiece.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

... (3 outputs left out)

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


model.safetensors: reconstructing file:   0%|          |  0.00B / 2.33GB            

model.safetensors: downloading bytes:           |  0.00B            

Sample 1 Generated Summary (mT5): জাতিসংঘের একটি কমিটি বলছে, চীনের জিনজিয়াং প্রদেশে উইগুর মুসলিমদের গণহারে ধরে বিভিন্ন বন্দীশিবিরে নেয়া হচ্ছে।
Sample 2 Generated Summary (mT5): ভারতের তামিলনাডু রাজ্যে কোকা-কোলা ও পেপসি নিষিদ্ধ করেছে ব্যবসায়ীরা।
Sample 3 Generated Summary (mT5): বাংলাদেশের রাজধানী ঢাকায় করোনাভাইরাসে আক্রান্ত হয়ে হাসপাতালে ভর্তি হওয়া রোগীর সংখ্যা প্রতিদিনই বাড়ছে।
Sample 4 Generated Summary (mT5): মার্কিন প্রেসিডেন্ট ডোনাল্ড ট্রাম্প অভিযোগ করেছেন সামাজিক মাধ্যম টুইটার আমেরিকার ২০২০ সালের নির্বাচনে নাক গলাচ্ছে।
Sample 5 Generated Summary (mT5): বাংলাদেশের বেশ কয়েকটি জেলা আগামী ২৪ ঘণ্টার মধ্যে বন্যার প্রকোপ ছড়িয়ে পড়ছে।
Sample 6 Generated Summary (mT5): ভারতের স্বরাষ্ট্রমন্ত্রী রাজনাথ সিং দাবি করেছেন, মাওবাদী বিদ্রোহ দমনে সরকারি কৌশলে বিরাট কোনও পরিবর্তন হচ্ছে না।
Sample 7 Generated Summary (mT5): বাংলাদেশে বিরোধীদল বিএনপি দেশের রাজনৈতিক পরিস্থিতি সম্পর্কে বিদেশি কূটনীতিকদের সাথে বৈঠক করেছে।
Sample 8 Generated Summary (mT5): বাংলাদেশে করোনাভাইরাস সংক্রমণের দ্বিতীয

### Evaluate mBART-50 Model
Now, we do the same pipeline using `facebook/mbart-large-50` config targeted for Bengali translation/summarization tasks.

In [7]:
# For mBART-50, we set generation configuration parameters to prevent repetitive loops during evaluation.
mbart_model_name = "facebook/mbart-large-50"

print(f"Loading tokenizer and model for {mbart_model_name} on {device}...")
mbart_tokenizer = AutoTokenizer.from_pretrained(mbart_model_name)
mbart_tokenizer.src_lang = "bn_IN"

mbart_model = AutoModelForSeq2SeqLM.from_pretrained(mbart_model_name).to(device)

def generate_summary_mbart(text, max_len=64):
    inputs = mbart_tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(device)
    outputs = mbart_model.generate(
        inputs["input_ids"],
        forced_bos_token_id=mbart_tokenizer.lang_code_to_id["bn_IN"],
        max_length=max_len,
        min_length=10,
        num_beams=4,
        repetition_penalty=3.0,  # Increased penalty to break the repetitive loops
        no_repeat_ngram_size=3,  # Prevents repeating 3-grams
        early_stopping=True
    )
    return mbart_tokenizer.decode(outputs[0], skip_special_tokens=True)

mbart_predictions = []
for i, sample in enumerate(eval_subset):
    pred = generate_summary_mbart(sample['text'])
    mbart_predictions.append(pred)
    print(f"Sample {i+1} Generated Summary (mBART):", pred)


Loading tokenizer and model for facebook/mbart-large-50 on cuda...


Loading weights:   0%|          | 0/519 [00:00<?, ?it/s]

Sample 1 Generated Summary (mBART): চীনের উষর প্রদেশ থেকে উদ্ব্ব্বশাব বয়েতে অপ্র বাতানা প্রতি অপরাধেখা বিনি বিভিন্ন দেশের আইন খসজন আও উদ্ আক কো ক না
Sample 2 Generated Summary (mBART): ভারতের তামিলনা অঞ্চলের রাজ্যে কোর প্রজনজনজন থেকে বিপিটি অপ্রতখাতে অন্যরেজেজা নাতাকোনো আক্রমগুলিই ই পরসন সময় অর্থ প্রতি অপরাধ বার্দ সম বচকু এ যে সবসরস্বযোনি দি ক খ
Sample 3 Generated Summary (mBART): ডেঙ্গস্বস্বস্বরসসসনবথচান্তান্তরেজেজতেপ্র প্রকোর্শাশত্রেরনা সোহমানোজনব্য প্রতিষ্ঠানে বেসরকারি হাসপাতালে সংরক্ষিত বসরসা অ সব অপরাধ তথ্য প্রতিচ্ছষরোডডিএম এম মেডিকেল সেন্টার সংস্থা তার না চগ
Sample 4 Generated Summary (mBART): বিক্রমটিটিচ আমেরিকারকম কণত অপ্র প্রোজনস্বস্বনকশতে না অপরাধের এ সব সময় যুক্তরাষ্ট্রের বুনে বা সিতাসফ থেকে উগ্রররপরোনো প্রতি সমরেনাথররপিরর পর শেষ হচ্ছেরর
Sample 5 Generated Summary (mBART): মধ্যের বসুরুসচান্তরে অপ্রেজনবজতে না আশককু প্রকার ব ব বনা ব বননি উ দেশের ব ব থেকে এ সব সময় ব বতাস্ব অপরাধ প্রতি ব ব হটি ব বকো কো ব ব অঞ্চল উপ গ সমো
Sample 6 Generated Summary (mBART): মাওর প্রেরুগজবলেতে

### Calculate Evaluation Metrics (ROUGE-1, ROUGE-2, ROUGE-L, and mBERT score)
We will compute the automatic evaluation metrics using `evaluate` or standard scorer packages to compare the two models.

In [9]:
from rouge_score import rouge_scorer
from bert_score import score as bert_score
import numpy as np
import pandas as pd

class BengaliTokenizer:
    def tokenize(self, text):
        return text.strip().split()

# Configure the scorer with our custom tokenizer object
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False, tokenizer=BengaliTokenizer())

def calculate_rouge(predictions, references):
    r1, r2, rl = [], [], []
    for pred, ref in zip(predictions, references):
        scores = scorer.score(ref, pred)
        r1.append(scores['rouge1'].fmeasure)
        r2.append(scores['rouge2'].fmeasure)
        rl.append(scores['rougeL'].fmeasure)
    return np.mean(r1) * 100, np.mean(r2) * 100, np.mean(rl) * 100

# Compute ROUGE scores
mt5_r1, mt5_r2, mt5_rl = calculate_rouge(mt5_predictions, references)
mbart_r1, mbart_r2, mbart_rl = calculate_rouge(mbart_predictions, references)

P_mt5, R_mt5, F1_mt5 = bert_score(mt5_predictions, references, lang="bn", verbose=False)
P_mbart, R_mbart, F1_mbart = bert_score(mbart_predictions, references, lang="bn", verbose=False)

results = {
    "Model": ["mT5 (csebuetnlp/multilingual)", "mBART-50 (facebook/mbart-large-50)"],
    "ROUGE-1 (%)": [mt5_r1, mbart_r1],
    "ROUGE-2 (%)": [mt5_r2, mbart_r2],
    "ROUGE-L (%)": [mt5_rl, mbart_rl],
    "mBERT F1 Score": [F1_mt5.mean().item(), F1_mbart.mean().item()]
}

df_results = pd.DataFrame(results)
display(df_results)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,Model,ROUGE-1 (%),ROUGE-2 (%),ROUGE-L (%),mBERT F1 Score
0,mT5 (csebuetnlp/multilingual),30.227937,10.835544,24.635777,0.786621
1,mBART-50 (facebook/mbart-large-50),2.907585,0.000000,2.907585,0.625741
